# Test the function to scrape Shot Chart From CHN

In [ ]:
import re
from typing import Optional
from bs4 import BeautifulSoup
import pandas as pd

# Map the site's raw data-type values into a simpler categorical outcome
SHOT_TYPE_MAP = {
    "goal": "GOAL",
    "shot": "SHOT_ON_GOAL",  # on net, but not goal
    "blocked": "BLOCKED",
    "wide": "MISSED",
    "pipe": "MISSED",
    "miss": "MISSED",
}

def _parse_xy_from_style(style_str: str):
    """
    Parse --x and --y CSS variables from a style attribute.
    e.g. ' --x: 85; --y: 36; ' -> (85.0, 36.0)
    """
    if not style_str:
        return None, None

    matches = dict(re.findall(r"--(x|y)\s*:\s*([0-9.]+)", style_str))
    x = float(matches.get("x")) if "x" in matches else None
    y = float(matches.get("y")) if "y" in matches else None
    return x, y


def extract_shot_chart_from_html(
    html: str,
    game_id: str,
    home_team: Optional[str] = None,
    away_team: Optional[str] = None,
) -> pd.DataFrame:
    """
    Extract shot chart data from a box score HTML page containing the <div id="shotxy"> block.

    Returns a DataFrame with columns:
        game_id, team_side, team,
        x_pct, y_pct,
        shot_type_raw, shot_outcome,
        period, oep,
        home_starts_right
    """
    soup = BeautifulSoup(html, "lxml")
    shot_root = soup.find("div", id="shotxy")
    if shot_root is None:
        # No shot chart on this page
        return pd.DataFrame()

    container = shot_root.find("div", class_="shotxy_container")
    if container is None:
        return pd.DataFrame()

    container_classes = container.get("class", [])
    home_starts_right = "home_starts_right" in container_classes

    rows = []

    for team_div in container.select("div.shotxy_team"):
        # home / away
        team_classes = team_div.get("class", [])
        if "home" in team_classes:
            side = "home"
        elif "away" in team_classes:
            side = "away"
        else:
            side = "unknown"

        # Team name from header if you don't pass them in
        h3 = team_div.find("h3")
        label = h3.get_text(strip=True) if h3 else ""
        # "Clarkson Shooting" -> "Clarkson"
        label = re.sub(r"\s+Shooting$", "", label)

        if side == "home" and home_team:
            team_name = home_team
        elif side == "away" and away_team:
            team_name = away_team
        else:
            team_name = label

        for span in team_div.find_all("span"):
            style_str = span.get("style", "")
            x_pct, y_pct = _parse_xy_from_style(style_str)

            shot_type_raw = span.get("data-type", "").strip().lower()
            period_str = span.get("data-period", "0")
            oep = span.get("data-oep")

            try:
                period = int(period_str)
            except ValueError:
                period = None

            shot_outcome = SHOT_TYPE_MAP.get(shot_type_raw, "UNKNOWN")

            rows.append(
                {
                    "game_id": game_id,
                    "team_side": side,
                    "team": team_name,
                    "x_pct": x_pct,
                    "y_pct": y_pct,
                    "shot_type_raw": shot_type_raw,
                    "shot_outcome": shot_outcome,
                    "period": period,
                    "oep": oep,
                    "home_starts_right": home_starts_right,
                }
            )

    return pd.DataFrame(rows)


In [ ]:
### ========================================== Test on a target game URL

## Game MSU at Notre Dame Friday Night
# url = "https://www.collegehockeynews.com/box/final/20251114/msu/ndm/"
url ="https://www.collegehockeynews.com/box/final/20251114/mic/psu/"

# --- Call Functrion to scrape Shot Chart From CHN ---
import requests
response = requests.get(url)
html = response.text
df_shot_chart = extract_shot_chart_from_html(
    html,
    game_id="20251114_psu_scUM",
    home_team="Michigan Dame",
    away_team="Penn State",
)
df_shot_chart.head()